# Unique OpWLS Photon Counts

This notebook reads the `tools::wcsv::ntuple` CSV files in `build/output`, filters rows with `ProcessName == "OpWLS"`, and counts unique photons by `(EventID, TrackID)`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("../build/output")
CSV_GLOB = "*.csv"
UNIQUE_KEYS = ["EventID", "TrackID"]
PROCESS_NAME = "OpWLS"
N_MUONS = 10

In [ ]:
def read_tools_wcsv(path: Path) -> pd.DataFrame:
    columns = []
    with path.open() as handle:
        for line in handle:
            if line.startswith("#column "):
                columns.append(line.strip().split()[-1])
            elif not line.startswith("#"):
                break

    if not columns:
        raise ValueError(f"No #column metadata found in {path}")

    df = pd.read_csv(path, comment="#", names=columns)
    df["SourceFile"] = path.name
    return df


def unique_wls_photons(df: pd.DataFrame) -> pd.DataFrame:
    required = set(UNIQUE_KEYS + ["ProcessName"])
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    filtered = df.loc[df["ProcessName"] == PROCESS_NAME].copy()
    return filtered.drop_duplicates(subset=UNIQUE_KEYS)


In [ ]:
csv_paths = sorted(OUTPUT_DIR.glob(CSV_GLOB))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {OUTPUT_DIR.resolve()}")

frames = [read_tools_wcsv(path) for path in csv_paths]
all_hits = pd.concat(frames, ignore_index=True)
all_unique_wls = unique_wls_photons(all_hits)

per_file_summary = []
for path, frame in zip(csv_paths, frames):
    unique_frame = unique_wls_photons(frame)
    per_file_summary.append(
        {
            "SourceFile": path.name,
            "Rows": len(frame),
            "OpWLSRows": int((frame["ProcessName"] == PROCESS_NAME).sum()),
            "UniqueOpWLSPhotons": len(unique_frame),
        }
    )

per_file_summary = pd.DataFrame(per_file_summary).sort_values("SourceFile").reset_index(drop=True)
per_file_summary

In [ ]:
combined_summary = pd.DataFrame(
    [
        {
            "TotalRows": len(all_hits),
            "TotalOpWLSRows": int((all_hits["ProcessName"] == PROCESS_NAME).sum()),
            "UniqueOpWLSPhotons": len(all_unique_wls),
        }
    ]
)
combined_summary

In [ ]:
sipm_volume_mask = all_hits["Volume"].astype(str).str.contains("sipm", case=False, na=False)
wls_sipm_hits = all_hits.loc[(all_hits["ProcessName"] == PROCESS_NAME) & sipm_volume_mask].copy()
unique_wls_seen_by_sipm = wls_sipm_hits.drop_duplicates(subset=UNIQUE_KEYS)

sipm_summary = pd.DataFrame(
    [
        {
            "UniqueOpWLSPhotonsSeenBySiPM": len(unique_wls_seen_by_sipm),
            "FractionOfUniqueOpWLSPhotons": len(unique_wls_seen_by_sipm) / len(all_unique_wls) if len(all_unique_wls) else 0.0,
        }
    ]
)
sipm_summary

In [ ]:
all_unique_wls[["SourceFile", "EventID", "TrackID", "ParentID", "OriginVolume", "Volume", "InitialEnergy"]].head(20)

In [ ]:
# Map SiPM copy numbers in final.gdml back to their lane indices and positions.
lane_start = -468.75
lane_pitch = 62.5
slab_half_xy = 500.0
sipm_pos_pos = 510.35

def sipm_copy_to_coords(copy_number: int) -> dict:
    copy_number = int(copy_number)
    if 100 <= copy_number <= 115:
        lane_index = copy_number - 100
        return {
            'Copynumber': copy_number,
            'Family': '+z row',
            'LaneIndex': lane_index,
            'x_mm': lane_start + lane_index * lane_pitch,
            'z_mm': sipm_pos_pos,
        }
    if 300 <= copy_number <= 315:
        lane_index = copy_number - 300
        return {
            'Copynumber': copy_number,
            'Family': '+x row',
            'LaneIndex': lane_index,
            'x_mm': sipm_pos_pos,
            'z_mm': lane_start + lane_index * lane_pitch,
        }
    return {
        'Copynumber': copy_number,
        'Family': 'unknown',
        'LaneIndex': -1,
        'x_mm': float('nan'),
        'z_mm': float('nan'),
    }

unique_wls_seen_by_sipm_per_copy = wls_sipm_hits.drop_duplicates(subset=UNIQUE_KEYS + ['Copynumber'])
sipm_counts = (
    unique_wls_seen_by_sipm_per_copy.groupby('Copynumber')
    .size()
    .rename('UniqueOpWLSPhotons')
    .reset_index()
)

sipm_coords = pd.DataFrame([sipm_copy_to_coords(copy_number) for copy_number in sipm_counts['Copynumber']])
sipm_heatmap = sipm_coords.merge(sipm_counts, on='Copynumber', how='left').sort_values(['Family', 'Copynumber']).reset_index(drop=True)
sipm_heatmap['PhotonsPerMuon'] = sipm_heatmap['UniqueOpWLSPhotons'] / N_MUONS

x_family = (
    sipm_heatmap.loc[sipm_heatmap['Family'] == '+z row', ['LaneIndex', 'x_mm', 'PhotonsPerMuon']]
    .sort_values('LaneIndex')
    .reset_index(drop=True)
)
z_family = (
    sipm_heatmap.loc[sipm_heatmap['Family'] == '+x row', ['LaneIndex', 'z_mm', 'PhotonsPerMuon']]
    .sort_values('LaneIndex')
    .reset_index(drop=True)
)

# Reconstruct a cell map by combining the two orthogonal SiPM responses.
# Each slab cell is assigned the average of the corresponding x-row and z-row
# SiPM counts, normalized by the number of muons in the run.
cell_values = 0.5 * (
    z_family['PhotonsPerMuon'].to_numpy()[:, np.newaxis]
    + x_family['PhotonsPerMuon'].to_numpy()[np.newaxis, :]
)

cell_edges = np.arange(-slab_half_xy, slab_half_xy + lane_pitch, lane_pitch)
cell_map_summary = pd.DataFrame(
    {
        'XRowMeanPhotonsPerMuon': x_family['PhotonsPerMuon'],
        'ZRowMeanPhotonsPerMuon': z_family['PhotonsPerMuon'],
    }
)

sipm_heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

mesh = ax.pcolormesh(
    cell_edges,
    cell_edges,
    cell_values,
    cmap='inferno',
    shading='flat',
    edgecolors='black',
    linewidth=0.25,
)

ax.set_title('Reconstructed Slab Cell Map from SiPM Counts')
ax.set_xlabel('x [mm]')
ax.set_ylabel('z [mm]')
ax.set_aspect('equal', adjustable='box')

cbar = plt.colorbar(mesh, ax=ax)
cbar.set_label('Mean unique OpWLS photons per muon')

plt.tight_layout()
plt.show()